# Distribution Planning Mean-Field Control Benchmark

Reference: Meunier, Pham & Reisinger, discrete-space benchmarks, Section "Distribution Planning" (`files/reference/discrete_benchmarks.tex`), based on Carmona et al.'s distribution-planning mean-field control problem.

**Model.** A population moves around the discrete torus $\mathcal X=\mathbb Z/10\mathbb Z=\{0,\ldots,9\}$, action space $\mathcal A=\{\mathrm{LEFT},\mathrm{STAY},\mathrm{RIGHT}\}$ shifting an agent by $-1,0,+1$ (mod 10). The transition kernel is **deterministic and population-independent**: unlike cybersecurity, the mean-field interaction here enters purely through the reward, which penalizes both movement and deviation from a fixed target law $\mu_\mathrm{target}=(0,0,0.05,0.10,0.20,0.30,0.20,0.10,0.05,0)$:
$$r(x,a,\mu) = -c_\mathrm{mov}|a| - \|\mu-\mu_\mathrm{target}\|_2^2, \qquad g(x,\mu) = -\|\mu-\mu_\mathrm{target}\|_2^2,$$
with $c_\mathrm{mov}=0.01$, $T=5$ for both training and validation (no train/val horizon split, unlike cybersecurity). The mismatch penalty is a smooth (quadratic) function of $\mu$, unlike two-state's $-\kappa|\mu(1)-(1-p)|$ penalty, which has a non-differentiable kink.

**Policy.** A population-dependent 2-hidden-layer MLP (width 256, $\tanh$), taking $(t,\mu)$ and outputting a $10\times3$ logit matrix, row-softmaxed: the same flat-parameter-packed pattern as cybersecurity's policy, just much larger (~76.6k parameters vs. ~1.5k), since the population state is 9-dimensional and the policy must coordinate movement across the whole state space.

As with cybersecurity, there is no known closed-form optimal policy, so this notebook has no "vs. optimal" diagnostics; instead it uses the reference's own distribution-planning-specific diagnostics: terminal/average $L_2$ mismatch to the target, the terminal transport discrepancy under the torus's cyclic metric ($W_{1,d_\mathrm{cyc}}$), and the expected cumulative movement.

**This notebook shows `main`-tier results automatically whenever they're available.** `configs/distribution_planning.py`'s `MAIN` uses the reference's exact protocol (5 seeds, $B=500$, $n_\mathrm{train}=100{,}000$, one $(T=5,\text{equal\_parameters},\text{exact})$ group, no horizon/budget sweep, unlike twostate/cybersecurity); everything below therefore works at either tier, mean $\pm$ std across seeds when `main` is available. `main`'s own training cost is far beyond what an inline notebook cell should launch; run `scripts/train_all.sh <workers> --env distribution_planning --alg <alg> --config main` (once per algorithm) to populate it.

In [ ]:
import sys
import time
from pathlib import Path

_notebook_start = time.perf_counter()

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
for path in (SRC, ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torch
import pandas as pd

torch.set_default_dtype(torch.float64)
torch.set_default_device("cuda" if torch.cuda.is_available() else "cpu")

from configs.distribution_planning import MAIN, MID
from mfc.environments.distribution_planning import LEFT, RIGHT, STAY, DistributionPlanning, DistributionPlanningConfig
from mfc.algorithms import simplex
from mfc.plotting import diagnostics as viz
from mfc.plotting.style import set_style
from scripts.train import run_all
from scripts.test import (
    average_mismatch_l2,
    cumulative_movement,
    cyclic_wasserstein,
    exact_gradient,
    exact_sensitivity_flow,
    generalization_eval,
    gradient_diagnostics,
    group_by,
    load_runs,
    logit_perturbation_coverage,
    objective_gap,
    oracle_gradient_estimate,
    perturbation_coverage,
    rollout,
    sensitivity_estimation_error,
    state_distribution,
    state_marginal_stability,
    terminal_mismatch_l2,
)

set_style()

DEFAULT_T = MID.horizons[0]  # 5: the only horizon this benchmark uses, at any tier

## Configuration and budget

Auto-detects whether `runs/distribution_planning/main/` has any saved runs and uses `main` if so (5 seeds, `equal_budget`/`particle`, $n_\mathrm{train}=100{,}000$; mfreinforce keeps the reference's $B=500$ anchor while simplex/reinforce are matched to its per-step transition budget at $B=15490$/$B=15500$), otherwise `mid` (1 seed, a reduced $B=100$/$n_\mathrm{train}=5{,}000$ for wall-clock time; a memory-practicality reduction given the policy MLP's ~76.6k parameters, not a fidelity choice -- see `configs/distribution_planning.py`'s module docstring). Unlike twostate/cybersecurity, there is only one $(T,\text{budget\_mode},\text{flow})$ group at either tier, so there is no horizon-scaling or budget/flow-comparison section here.

In [ ]:
main_dir = ROOT / "runs" / "distribution_planning" / "main"
tier = "main" if list(main_dir.glob("*_seed*.pt")) else "mid"
cfg = MAIN if tier == "main" else MID

print(f"tier: {tier}")
print(f"algorithms:    {cfg.algorithms}")
print(f"lambdas:       {cfg.lambdas}  (simplex perturbation scale)")
print(f"epsilon:       {cfg.epsilon}  (logit perturbation scale, fixed -- not swept like lambda)")
print(f"T={cfg.horizons[0]} (training and validation)")
print(f"seeds:         {cfg.seeds}")
print(f"B={cfg.B}, n_aux={cfg.n_aux}, sigma={cfg.sigma}, lr={cfg.lr}, n_train={cfg.n_train}")
print(f"mu0 ~ Dirichlet(1,...,1) during training; validation mu0={cfg.mu0_val}")
print(f"target law: {DistributionPlanningConfig().target_law}")
if tier == "mid":
    print("\n(no runs/distribution_planning/main/ data yet, showing mid-tier results; run scripts/train_all.sh "
          "<workers> --env distribution_planning --alg <alg> --config main, once per algorithm, for the reference's "
          "full protocol and real seed-to-seed statistics)")

## Train (or load cached results)

At `mid`, trains first if nothing is cached yet. At `main`, only loads what's already there: training the reference's full protocol (5 seeds x $n_\mathrm{train}=100{,}000$) is far too expensive for an inline notebook cell. Adapts `env`'s dtype to match whatever's loaded.

In [ ]:
env = DistributionPlanning()
runs_dir = ROOT / "runs" / "distribution_planning" / tier

runs = []
for alg in cfg.algorithms:
    if tier == "mid" and not list(runs_dir.glob(f"{alg}_*_seed*.pt")):
        run_all("distribution_planning", alg, "mid")
    runs += load_runs("distribution_planning", alg, tier)

if runs and runs[0]["theta_final"].dtype != env.dtype:
    run_dtype = runs[0]["theta_final"].dtype
    print(f"note: loaded runs are {run_dtype}, switching env and the notebook's default dtype to match (was {env.dtype})")
    torch.set_default_dtype(run_dtype)
    env = DistributionPlanning(dtype=run_dtype)

total_train_seconds = sum(r["elapsed_seconds"] for r in runs)
print(f"{len(runs)} runs loaded ({tier}); total training compute time: {total_train_seconds:.1f}s ({total_train_seconds / 60:.1f} min)")

## Default group

`by_lambda` picks seed 0 as a representative $\theta$ per $\lambda$; `by_lambda_all_seeds` keeps every seed for the aggregate diagnostics, which cover every $\lambda$ throughout this notebook, not just one.

In [ ]:
mu0_val = torch.tensor(cfg.mu0_val, dtype=env.dtype, device=env.device)

simplex_runs = [r for r in runs if r["alg"] == "simplex"]
mfreinforce_runs = [r for r in runs if r["alg"] == "mfreinforce"]
reinforce_runs = [r for r in runs if r["alg"] == "reinforce"]

by_lambda_all_seeds = {lam: grp for (lam,), grp in group_by(simplex_runs, "lam").items()}
by_lambda = {lam: next((r for r in grp if r["seed"] == 0), grp[0]) for lam, grp in by_lambda_all_seeds.items()}
theta_02 = by_lambda[0.2]["theta_final"]

print(f"{len(simplex_runs)} simplex runs across {len(by_lambda)} lambda values, "
      f"{len(mfreinforce_runs)} mfreinforce run(s), {len(reinforce_runs)} reinforce run(s)")

## Evolution of the validation reward

The exact validation objective $J_T(\theta_m;\mu_0^\mathrm{val})$ every 10 training iterations, one line per simplex $\lambda$ plus reinforce and mfreinforce. At `main`, each line is the mean $\pm$ 1 std across 5 seeds.

In [ ]:
fig, ax = viz.plot_validation_curve(runs)
ax.set_title(f"Validation objective by training iteration ({tier} tier)")

## State distribution over time

The learned policy's population flow $\mu_t^\theta$ from $\mu_0^\mathrm{val}$ (the uniform law), for every $\lambda$ (seed 0), against the fixed target law. The graph below shows $\lambda=0.2$ specifically; the table gives every $\lambda$'s terminal-state distribution $\mu_T^\theta$.

In [ ]:
mu_flow = state_distribution(env, env.policy_probs, theta_02, mu0_val, DEFAULT_T)
fig, ax = viz.plot_state_distribution(mu_flow, target_law=env.target_law, state_labels=[str(x) for x in range(env.n_states)])
ax.set_title("Population flow toward the target distribution")

rows = []
for lam in sorted(by_lambda):
    flow = state_distribution(env, env.policy_probs, by_lambda[lam]["theta_final"], mu0_val, DEFAULT_T)
    rows.append({"λ": lam, **{f"mu_T({x})": flow[-1, x].item() for x in range(env.n_states)}})
rows.append({"λ": "target", **{f"mu_T({x})": env.target_law[x].item() for x in range(env.n_states)}})
pd.DataFrame(rows).set_index("λ")

## Population-mismatch and movement diagnostics

Across the full $\lambda$ sweep, at each $\lambda$'s own learned $\theta$ (reference "Evaluation criteria"), plus mfreinforce and reinforce for comparison (mean $\pm$ std across seeds at `main`, a single value at `mid`):
- $\mathcal E_T^{(2)}=\|\mu_T^\theta-\mu_\mathrm{target}\|_2$ (terminal $L_2$ mismatch) and $\bar{\mathcal E}^{(2)}=\frac1{T+1}\sum_t\|\mu_t^\theta-\mu_\mathrm{target}\|_2$ (average over the episode).
- $\mathcal E_T^{(W)}=W_{1,d_\mathrm{cyc}}(\mu_T^\theta,\mu_\mathrm{target})$, the terminal transport discrepancy under the torus's cyclic metric: distinguishes a small spatial displacement from a Euclidean-comparable mismatch at distant sites.
- $\mathcal C_\mathrm{mov}=\sum_{t<T}\sum_x\mu_t(x)[1-\pi_t(\mathrm{STAY}\mid x,\mu_t)]$, the expected cumulative movement.

In [ ]:
def mismatch_row(label, group):
    E_T2 = torch.tensor([terminal_mismatch_l2(env, env.policy_probs, r["theta_final"], mu0_val, DEFAULT_T).item() for r in group])
    E_bar2 = torch.tensor([average_mismatch_l2(env, env.policy_probs, r["theta_final"], mu0_val, DEFAULT_T).item() for r in group])
    E_TW = torch.tensor([cyclic_wasserstein(state_distribution(env, env.policy_probs, r["theta_final"], mu0_val, DEFAULT_T)[-1], env.target_law).item() for r in group])
    C_mov = torch.tensor([cumulative_movement(env, env.policy_probs, r["theta_final"], mu0_val, DEFAULT_T, stay_action=STAY).item() for r in group])
    return {
        "theta": label,
        "E_T^(2)": E_T2.mean().item(), "E_T^(2) std": E_T2.std().item() if len(E_T2) > 1 else 0.0,
        "E_bar^(2)": E_bar2.mean().item(), "E_bar^(2) std": E_bar2.std().item() if len(E_bar2) > 1 else 0.0,
        "E_T^(W)": E_TW.mean().item(), "E_T^(W) std": E_TW.std().item() if len(E_TW) > 1 else 0.0,
        "C_mov": C_mov.mean().item(), "C_mov std": C_mov.std().item() if len(C_mov) > 1 else 0.0,
    }

rows = [mismatch_row(f"λ={lam}", grp) for lam, grp in sorted(by_lambda_all_seeds.items())]
rows.append(mismatch_row("mfreinforce", mfreinforce_runs))
rows.append(mismatch_row("reinforce", reinforce_runs))
pd.DataFrame(rows).set_index("theta")

## $J^\lambda$ vs $J$, and gradient bias/variance

The simplex plug-in gradient estimator (`mfc.algorithms.simplex.gradient_estimate`) is

$$\hat g_{B,n,\lambda,\eta}(\theta) = \frac1B\sum_{b=1}^B\sum_{t=0}^{T}\Big[\mathbb 1_{\{t<T\}}L_t^{(b)} + Q_t^{(b)}\Big]\,G_t^{(b)},$$

with $L_t=\nabla_\theta\log\pi_t^\theta(a_t\mid x_t,M_t)$ the direct policy score, $Q_t=-\frac{1-\lambda}{\lambda}H(q_t)^\top\hat D_t$ the population-sensitivity correction, and $G_t$ the return-to-go. Both diagnostics below are at every $\lambda$'s own learned $\hat\theta_\lambda$ (seed 0), summarized as $\|\text{bias}\|$/$\|\text{std}\|$ over the full ~76.6k-dimensional MLP parameter vector (a per-component view isn't legible at this dimension); only simplex has this plug-in estimator, so this table stays simplex-only.

In [ ]:
gaps, grad_diag = {}, {}
for lam, r in by_lambda.items():
    theta = r["theta_final"]
    gaps[lam] = objective_gap(env, env.policy_probs, theta, mu0_val, DEFAULT_T, lam=lam, sigma=cfg.sigma, n_samples=5000)
    grad_diag[lam] = gradient_diagnostics(
        env, env.policy_probs, theta, mu0_val, DEFAULT_T,
        lam=lam, n_aux=cfg.n_aux, B=cfg.B, sigma=cfg.sigma, reps=10,
    )

rows = []
for lam in sorted(gaps):
    g, d = gaps[lam], grad_diag[lam]
    rows.append({
        "λ": lam,
        "J(theta_hat)": g["J"].item(), "J^lambda(theta_hat) (MC)": g["J_lambda_mean"].item(), "|gap|": abs(g["gap"].item()),
        "||bias||": d["bias"].norm().item(), "||std||": d["std"].norm().item(),
    })
pd.DataFrame(rows).set_index("λ")

## Perturbation coverage: simplex $d_{TV}(M^\lambda,\mu)\le\lambda$ and mfreinforce $\mathbb E[d_{TV}]\le\varepsilon/2$

Checked (as elsewhere) by direct sampling at a few representative $N=10$-dimensional population laws: the validation (uniform) law, the target law itself, and a point from the learned flow. Simplex's bound holds *almost surely* (every draw); mfreinforce's logit perturbation only holds *in expectation* (`files/Discrete RL - Meunier, Pham, Reisinger.md`, Lemma 2.2).

In [ ]:
mu_labels = ["mu0_val (uniform)", "target_law", "mu_flow[2]"]
mu_samples = torch.stack([mu0_val, env.target_law, mu_flow[2]])

coverage = perturbation_coverage(mu_samples, lam=0.2, sigma=cfg.sigma, n_samples=5000)
for r in coverage:
    assert r["within_bound"], "the perturbation theorem's bound should never be violated"
print("simplex, lambda=0.2:")
display(pd.DataFrame([{"mu": lbl, "mean_dTV": r["mean_dTV"].item(), "max_dTV": r["max_dTV"].item(), "bound (lambda)": 0.2, "within_bound": r["within_bound"]}
                       for lbl, r in zip(mu_labels, coverage)]).set_index("mu"))
fig, ax = viz.plot_perturbation_coverage(coverage, 0.2, mu_labels=mu_labels)
ax.set_title("Simplex perturbation coverage against the TV bound")

logit_coverage = logit_perturbation_coverage(mu_samples, epsilon=cfg.epsilon, n_samples=5000)
for r in logit_coverage:
    assert r["within_bound"], "Lemma 2.2's expected-value bound should hold at this sample size"
print("\nmfreinforce, epsilon={}:".format(cfg.epsilon))
display(pd.DataFrame([{"mu": lbl, "mean_dTV": r["mean_dTV"].item(), "max_dTV": r["max_dTV"].item(), "bound (epsilon/2)": cfg.epsilon / 2, "within_bound (mean)": r["within_bound"]}
                       for lbl, r in zip(mu_labels, logit_coverage)]).set_index("mu"))

## Initial, target, and terminal distributions

The reference's own qualitative check (reference "Evaluation criteria"): does a high validation value come from genuine transport toward the target, or from excessive local oscillation?

In [ ]:
fig, ax = viz.plot_distribution_comparison(
    {"initial (mu0_val)": mu0_val, "target": env.target_law, "terminal (learned)": mu_flow[-1]},
    state_labels=[str(x) for x in range(env.n_states)],
)
ax.set_title("Initial, target, and learned terminal distributions")

## Sample trajectory under the learned policy

One sampled state trajectory ($\lambda=0.2$, seed 0) from $\mu_0^\mathrm{val}$, over $T=5$ steps on the torus.

In [ ]:
learned_traj = rollout(env, env.policy_probs, theta_02, mu0_val, T=DEFAULT_T, generator=torch.Generator(device=env.device).manual_seed(0))
fig, ax = viz.plot_trajectories(learned_traj)
ax.set_title("Sample state path under the learned policy")

## Generalization without retraining

Evaluating every $\lambda$'s learned $\hat\theta_\lambda$ (seed 0) exactly (no retraining) under different initial laws, a longer horizon, and a stronger/weaker movement penalty.

In [ ]:
corner = torch.zeros(env.n_states, dtype=env.dtype, device=env.device)
corner[0] = 1.0
scenarios = [
    {"name": "baseline (mu0_val)"},
    {"name": "mu0=all mass at 0", "mu0": corner},
    {"name": "mu0=target_law", "mu0": env.target_law},
    {"name": "T=10", "T": 10},
    {"name": "2x movement penalty", "env": DistributionPlanning(DistributionPlanningConfig(c_mov=0.02), dtype=env.dtype, device=env.device)},
    {"name": "no movement penalty", "env": DistributionPlanning(DistributionPlanningConfig(c_mov=0.0), dtype=env.dtype, device=env.device)},
]

rows = {sc["name"]: {"scenario": sc["name"]} for sc in scenarios}
for lam in sorted(by_lambda):
    gen_results = generalization_eval(env, env.policy_probs, by_lambda[lam]["theta_final"], mu0_val, DEFAULT_T, scenarios)
    for res in gen_results:
        rows[res["name"]][f"λ={lam}"] = res["J"].item()
pd.DataFrame(list(rows.values())).set_index("scenario")

## Comparing the three algorithms

Final validation objective for each algorithm: every simplex $\lambda$, reinforce (`mfc.algorithms.reinforce`, which omits the population-sensitivity correction $Q_t(D_t)$ entirely), and mfreinforce. Mean $\pm$ std across seeds at `main`; a single value at `mid` (which can't separate genuine differences from noise).

In [ ]:
rows = []
for lam, group in sorted(by_lambda_all_seeds.items()):
    vals = torch.tensor([r["validation_J"][-1].item() for r in group])
    rows.append({"algorithm": f"simplex λ={lam}", "final validation J": vals.mean().item(), "std": vals.std().item() if len(vals) > 1 else 0.0, "n_seeds": len(vals)})
for alg, group in (("mfreinforce", mfreinforce_runs), ("reinforce", reinforce_runs)):
    vals = torch.tensor([r["validation_J"][-1].item() for r in group])
    rows.append({"algorithm": alg, "final validation J": vals.mean().item(), "std": vals.std().item() if len(vals) > 1 else 0.0, "n_seeds": len(vals)})
pd.DataFrame(rows).set_index("algorithm")

## Additional statistical validation (discrete-state theory)

The sections above check the estimators against each other and against training outcomes. The sections below instead check the theory's own asymptotic claims from `files/reference/discrete_state_space(2).tex` directly, at a single fixed $\theta=\hat\theta_{0.2}$ (`theta_02`) so $\lambda$ is the only thing varying. Like two-state (and unlike cybersecurity), distribution planning's mean-field coupling lives entirely in the **reward**: `DistributionPlanning.transition_probs` is independent of $\mu$. Unlike two-state, the mismatch penalty $\|\mu-\mu_\mathrm{target}\|_2^2$ is smooth (quadratic) rather than having a non-differentiable kink, so gradient-level convergence is not expected to fail here for the same reason it did for two-state.

This MLP-policy environment is far more expensive per Monte Carlo replicate than two-state's lookup-table policy (a forward+backward pass through a ~76.6k-parameter network per replicate), so sample sizes below are smaller than two-state's notebook uses; standard errors (SE) are still reported throughout so every point estimate's precision is explicit rather than assumed.

### Convergence of the perturbed objective: $|J^\lambda(\hat\theta_{0.2})-J(\hat\theta_{0.2})|=O(\lambda)$

Theorem "Convergence of the perturbed objective": $|J^\lambda(\theta)-J(\theta)|\le C_T\lambda$ for every $\theta$, with $C_T$ independent of $\lambda$. `n_samples=200,000` (cheap: this is a single batched Monte Carlo evaluation, not a Python-level loop).

In [ ]:
gaps_ref = {lam: objective_gap(env, env.policy_probs, theta_02, mu0_val, DEFAULT_T, lam=lam, sigma=cfg.sigma, n_samples=200_000) for lam in cfg.lambdas}

rows = []
for lam in sorted(gaps_ref):
    g = gaps_ref[lam]
    gap, se = g["gap"].item(), g["J_lambda_se"].item()
    rows.append({"λ": lam, "J(theta_02)": g["J"].item(), "J^lambda(theta_02) (MC)": g["J_lambda_mean"].item(), "SE": se, "|gap|": abs(gap), "|gap|/SE": abs(gap) / se, "|gap|/lambda": abs(gap) / lam})
pd.DataFrame(rows).set_index("λ")

### Gradient-level convergence: $\|\nabla J^\lambda(\hat\theta_{0.2})-\nabla J(\hat\theta_{0.2})\|=O(\lambda)$

Theorem "Gradient-level convergence" (needs the additional smoothness of Assumption "Smoothness of the averaged dynamics": $R_t^\theta$ continuously differentiable in $\mu$, satisfied here since the mismatch penalty is quadratic). The *oracle-D* plug-in estimator (`simplex.gradient_estimate` fed the exact sensitivity flow `exact_sensitivity_flow` instead of the auxiliary plug-in estimate) satisfies $\mathbb E[\hat g^{\mathrm{orc}}_{B,\lambda}(\theta)]=\nabla_\theta J^\lambda(\theta)$ exactly, isolating the perturbation-bias term from sensitivity-estimation noise. `B=cfg.B`, `reps=30` (smaller than two-state's 500, given the per-replicate MLP cost).

In [ ]:
reps_grad = 30
exact_grad_ref = exact_gradient(env, env.policy_probs, theta_02, mu0_val, DEFAULT_T)
oracle_samples_ref, oracle_mean_ref = {}, {}
for lam in cfg.lambdas:
    samples = torch.stack([
        oracle_gradient_estimate(env, env.policy_probs, theta_02, mu0_val, DEFAULT_T, lam=lam, sigma=cfg.sigma, B=cfg.B)
        for _ in range(reps_grad)
    ])
    oracle_samples_ref[lam] = samples
    oracle_mean_ref[lam] = samples.mean(dim=0)

rows = []
for lam in sorted(oracle_mean_ref):
    bias_vec = oracle_mean_ref[lam] - exact_grad_ref
    bias = bias_vec.norm().item()
    bias_se = (oracle_samples_ref[lam].std(dim=0) / reps_grad**0.5).norm().item()
    rows.append({"λ": lam, "||grad J - grad J^lambda||": bias, "SE": bias_se, "bias/SE": bias / bias_se, "bias/lambda": bias / lam})
pd.DataFrame(rows).set_index("λ")

### Population-flow sensitivity estimator: $\hat D_t(k)\to D_t^\theta(k)$

`simplex.estimate_sensitivity_flow`'s single-batch forward estimator $\hat D_t(k)$ of $D_t^\theta(k)=\nabla_\theta\mu_t^\theta(k)$, against the exact value (`exact_sensitivity_flow`, autograd). Bias $A_\eta$ is a property of $\eta$ alone (the single-batch estimator is unbiased for $D_t^{\eta,\theta}$ at *any* $n$, by the conditional-centering Remark); variance $V_\eta/n$ is a property of $n$ alone. Since distribution planning's *policy* (the MLP) genuinely depends on $\mu$ even though the transition kernel does not, there is no structural reason to expect $A_\eta=0$ here, unlike two-state's fully $\mu$-independent policy.

In [ ]:
reps_sens = 100
sens_by_eta = {eta: sensitivity_estimation_error(env, env.policy_probs, theta_02, mu0_val, DEFAULT_T, eta=eta, n=100, sigma=cfg.sigma, reps=reps_sens) for eta in cfg.lambdas}
rows = [{"eta": eta, "n": 100, "bias_norm": r["bias_norm"].sum().item(), "bias_se": r["bias_se"].sum().item(), "resolved (bias>2*SE)": bool(r["bias_norm"].sum().item() > 2 * r["bias_se"].sum().item())} for eta, r in sorted(sens_by_eta.items())]
print(f"bias vs. eta, at n=100 (large relative to cfg.n_aux={cfg.n_aux}, so V_eta/n is small and A_eta is precisely resolved):")
display(pd.DataFrame(rows).set_index("eta"))

n_values = sorted({cfg.n_aux, 5 * cfg.n_aux, 20 * cfg.n_aux})
sens_by_n = {n: sensitivity_estimation_error(env, env.policy_probs, theta_02, mu0_val, DEFAULT_T, eta=0.2, n=n, sigma=cfg.sigma, reps=reps_sens) for n in n_values}
rows = [{"n": n, "eta": 0.2, "bias_norm": r["bias_norm"].sum().item(), "bias_se": r["bias_se"].sum().item(), "variance": r["variance"].sum().item(), "mse": r["mse"].sum().item()} for n, r in sorted(sens_by_n.items())]
print(f"\nvariance vs. n, at fixed eta=0.2 (cfg.n_aux={cfg.n_aux} is the actual training value):")
display(pd.DataFrame(rows).set_index("n"))

### Gradient-estimator bias decomposition: (I) Monte Carlo + (II) sensitivity-estimation + (III) perturbation

Proposition "Mean of the main-batch estimator" decomposes $\hat g_{B,n,\lambda,\eta}(\theta)-\nabla_\theta J(\theta)$ into (I) zero-mean main-batch Monte Carlo noise, (II) the bias from plugging in $\hat D_t$ instead of the exact $D_t^\theta$, and (III) the perturbation bias $\nabla J^\lambda-\nabla J$ above. The ordinary plug-in samples (`gradient_diagnostics`, `n_aux=cfg.n_aux`, `B=cfg.B`, `reps=30`) mix (II) and (III); the oracle-D estimator above isolates (III); their difference approximates (II).

In [ ]:
plugin_ref = {lam: gradient_diagnostics(env, env.policy_probs, theta_02, mu0_val, DEFAULT_T, lam=lam, n_aux=cfg.n_aux, B=cfg.B, sigma=cfg.sigma, reps=reps_grad) for lam in cfg.lambdas}

rows = []
for lam in sorted(oracle_mean_ref):
    term3 = (oracle_mean_ref[lam] - exact_grad_ref).norm().item()
    term23 = plugin_ref[lam]["bias"].norm().item()
    term2_vec = plugin_ref[lam]["mean_estimate"] - oracle_mean_ref[lam]
    term2 = term2_vec.norm().item()
    term2_se = (((plugin_ref[lam]["std"] ** 2 + oracle_samples_ref[lam].std(dim=0) ** 2) / reps_grad).sum() ** 0.5).item()
    rows.append({"λ": lam, "perturbation bias (III)": term3, "(II) approx": term2, "(II) SE": term2_se, "(II)/SE": term2 / term2_se, "total plug-in bias (II)+(III)": term23})
pd.DataFrame(rows).set_index("λ")

### Stability of the perturbed state marginal: $d_{TV}(\nu_t^{\lambda,\theta},\mu_t^\theta)$

Lemma "Stability of the state marginal": $d_{TV}(\nu_t^{\lambda,\theta},\mu_t^\theta)\le L_K\lambda t$, where $\nu_t^{\lambda,\theta}:=\mathrm{Law}(X_t^{\lambda,\theta})$ is the law of the *perturbed* state process (fresh $q_t$ redrawn at every step) and $\mu_t^\theta$ the exact nominal flow. `DistributionPlanning.transition_probs` is independent of $\mu$, but the *policy* is not (the MLP takes $\mu$ as an input), so $K_t^\theta(\cdot\mid i,m)=\sum_a\pi_t^\theta(a\mid i,m)P_t(j\mid i,a)$ still genuinely depends on $m$ through $\pi_t^\theta$ alone: $L_K$ need not be $0$ here, unlike two-state (whose *policy* was also $\mu$-independent). `n_samples=100,000` per $(\lambda,t)$.

In [ ]:
rows = []
for lam in sorted(by_lambda):
    tv = state_marginal_stability(env, env.policy_probs, by_lambda[lam]["theta_final"], mu0_val, T=DEFAULT_T, lam=lam, sigma=cfg.sigma, n_samples=100_000)
    rows.append({"λ": lam, **{f"t={t}": tv[t].item() for t in range(tv.shape[0])}})
pd.DataFrame(rows).set_index("λ")

## Summary

Total notebook runtime (including any training performed in this run):

In [ ]:
print(f"total notebook runtime: {time.perf_counter() - _notebook_start:.1f}s")